In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

# Ativa o tracing do LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY") or getpass.getpass("LangSmith API Key: ")
os.environ["LANGSMITH_PROJECT"] = "rag-curso-alura"  # ou o nome do projeto que preferir


In [2]:
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader

pdfs = DirectoryLoader(r"G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos", glob="*.pdf", loader_cls=PyPDFLoader).load()


C:\Users\MarcosRibeiro\AppData\Local\Temp\ipykernel_47084\552424176.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
g:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
len(pdfs)

59

In [4]:
from transformers import AutoTokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [6]:
from langchain_text_splitters import CharacterTextSplitter

splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer, chunk_size=1250, chunk_overlap=150
)

In [7]:
pedacos = splitter.split_documents(pdfs)

In [8]:
len(pedacos)

59

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="bge-m3:567m")

vector_store = FAISS.from_documents(
    documents=pedacos, embedding=embeddings
)

In [10]:
retriever = vector_store.as_retriever()

In [11]:
from langchain_ollama.llms import OllamaLLM

modelo = OllamaLLM(model="gemma3:4b")

In [12]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [13]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "Responda sempre em português brasileiro usando exclusivamente o conteúdo fornecido.\n\nContexto:\n{contexto}"),
        ("human", "{query}")
    ]
)


In [14]:
from langchain_core.output_parsers import StrOutputParser

cadeia = prompt | modelo | StrOutputParser()

In [15]:
pergunta = "Como fazer um seguro viagem?"

trechos = retriever.invoke(pergunta)
for i, trecho in enumerate(trechos, 1):
    origem = trecho.metadata.get("source", "Desconhecido")
    pagina = trecho.metadata.get("page", 0)
    print(f"--- Trecho {i} | Arquivo: {origem} | Página: {pagina} ---")
    print(trecho.page_content[:300]) # Primeiros 300 caracteres
    print("\n")

contexto = "\n\n".join(trecho.page_content for trecho in trechos)

cadeia.invoke({"query": pergunta, "contexto": contexto})


--- Trecho 1 | Arquivo: G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos\GTB_platinum_Nov23.pdf | Página: 21 ---
Versão: novembro/2021  
feita se não é superior ao custo médio de tais serviços e fornecimentos na localidade onde  receberam, 
considerando a natureza e a gravidade da Doença Súbita ou Acidente no relação com os quais esses 
serviços e fornecimentos são recebidos. 
Viagem Segurada: É o período de t


--- Trecho 2 | Arquivo: G:\Meu Drive\01-Alura\AIEngineering\05 - LangChain Técnicas Avançadas de RAG\4910-LangChain-Tecnicas-Avancas-de-RAG\documentos\GTB_platinum_Nov23.pdf | Página: 13 ---
Versão: novembro/2021  
Aspectos Importantes: 
- 
- As viagens estão cobertas por um período máximo de 31 (Trinta e um) dias consecutivos a partir 
da primeira data de embarque de cada viagem. 
- As Despesas Médicas estão cobertas até o valor máximo de benefício de USD† 25.000 por Pessoa 
Elegível. 


--- Trecho 3

'Para fazer um seguro viagem, você precisa seguir alguns passos e atender a certos requisitos, conforme as informações fornecidas:\n\n1.  **Verifique a Elegibilidade:** Certifique-se de que você se qualifica para o programa MasterAssist Plus, que é oferecido aos portadores de cartões Mastercard Platinum™ e seus dependentes.\n2.  **Emita o Bilhete de Seguro:** Acesse o portal www.aig.com/Mastercard/pt e emita o seu Bilhete de Seguro Viagem. Este documento é essencial para ter cobertura. O Bilhete tem vigência de 12 meses a partir da data da emissão, e somente Viagens Cobertas ocorridas após essa data serão consideradas.\n3.  **Pagamento da Passagem:** Para ser elegível à cobertura, você deve pagar o custo total da passagem do Transporte Público Autorizado com seu cartão Mastercard Platinum™ ou com pontos ganhos em um Programa de Recompensas associado ao seu cartão.\n4.  **Cobertura:** A cobertura cobre despesas médicas e hospitalares em viagens ao exterior (acidentes ou doenças súbitas)

In [16]:
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {
        "contexto": retriever | format_docs,
        "query": RunnablePassthrough()
    }
    | prompt
    | modelo
    | StrOutputParser()
)


In [17]:
rag_chain.invoke(pergunta)

'Para fazer um seguro viagem, siga estes passos, com base nas informações fornecidas:\n\n1.  **Verifique a Elegibilidade:** Certifique-se de que você se enquadra nos critérios para cobertura do seguro. Isso inclui ter um cartão Mastercard Platinum™ e seus dependentes (cônjuge, companheiro(a) e filhos dependentes) viajando juntos ou separados, e que o custo total da passagem de transporte público autorizado seja cobrado do seu cartão Mastercard Platinum™ ou adquirido com pontos ganhos em um Programa de Recompensas associado.\n\n2.  **Emita o Bilhete de Seguro:** Acesse o portal www.aig.com/Mastercard/pt para emitir o Bilhete de Seguro Viagem. É fundamental que o bilhete seja emitido antes da viagem e apresentado no caso de qualquer ocorrência. O bilhete tem vigência de 12 meses a partir da data de emissão, e somente Viagens Cobertas ocorridas após essa emissão serão cobertas.\n\n3.  **Entenda as Coberturas:** O seguro oferece diversas coberturas, incluindo:\n    *   Despesas Médicas e H

In [18]:
query_model = OllamaLLM(model="gemma3:1b")

In [19]:
rewriter_prompt_template = """
Gere consulta de pesquisa para o banco de dados de vetores (Vector DB) a partir de uma pergunta do usuário,
permitindo uma resposta mais precisa por meio da busca semantica.
Basta retornar a consulta revisada do Vector DB, entre aspas.

Pergunta do usuário: {user_question}

Consulta revisada do Vector DB:
"""

In [20]:
from langchain_core.prompts import PromptTemplate

# Transforma a string em um PromptTemplate do LangChain
rewriter_prompt = PromptTemplate.from_template(rewriter_prompt_template)

# Agora sim você pode encadear com o pipe (|)
rewriter_chain = rewriter_prompt | query_model | StrOutputParser()


In [21]:
print(pergunta)
rewriter_chain.invoke({"user_question": pergunta})

Como fazer um seguro viagem?


'```\n"seguro viagem" \n```\n'

In [22]:
rewriter_rag_chain = (
    {
        "contexto": RunnablePassthrough() | rewriter_chain | retriever | format_docs,
        "query": RunnablePassthrough()
    }
    | prompt
    | modelo
    | StrOutputParser()
)

In [23]:
rewriter_rag_chain.invoke(pergunta)

'Para fazer um seguro viagem com o MasterAssist Plus, siga estes passos:\n\n1.  **Verifique a elegibilidade:** Você precisa ser portador do cartão Mastercard Platinum™ ou ter um dependente (cônjuge ou companheiro(a) e filhos dependentes) que seja, e ambos devem estar incluídos no Bilhete de Seguro Anual.\n2.  **Atualize o Bilhete de Seguro:** Se houver alguma mudança nos seus dependentes (cônjuge ou companheiro(a) / filho(s)), você deve reemitir seu Bilhete de Seguro atualizado.\n3.  **Obtenha a cobertura:** As coberturas se aplicam sujeitas aos termos e condições se a perda coberta ocorrer durante a vigência do Bilhete de Seguro, desde que o custo total da passagem de um Transporte Público Autorizado seja cobrado do seu cartão Mastercard Platinum TM elegível e/ou adquirida com pontos ganhos em um Programa de Recompensas associado ao seu cartão Mastercard PlatinumTM.\n4.  **Emita o Bilhete de Seguro:** É imprescindível que você emita o Bilhete de Seguro através do portal www.aig.com/Ma